In [0]:
%sql
use catalog `samples`; 
select * from `nyctaxi`.`trips` limit 100;

In [0]:
df = spark.table("samples.nyctaxi.trips")
display(df.limit(100))

In [0]:
from pyspark.sql.functions import col

df_selected = df.select(
    col("tpep_pickup_datetime"),
    col("tpep_dropoff_datetime"),
    col("trip_distance"),
    col("fare_amount"),
    col("pickup_zip"),
    col("dropoff_zip")
)

In [0]:
df_clean = df_selected.filter(
    (col("trip_distance") > 0) &
    (col("fare_amount") > 0) &
    col("tpep_pickup_datetime").isNotNull() &
    col("tpep_dropoff_datetime").isNotNull()
)

In [0]:
from pyspark.sql.functions import unix_timestamp

df_enriched = df_clean.withColumn(
    "trip_minutes",
    (unix_timestamp("tpep_dropoff_datetime") - 
     unix_timestamp("tpep_pickup_datetime")) / 60
)

In [0]:
df_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.silver_nyc_taxi_trips")


In [0]:
df = spark.table("workspace.default.silver_nyc_taxi_trips")
display(df)

In [0]:
from pyspark.sql.functions import avg

result = spark.table("workspace.default.silver_nyc_taxi_trips") \
    .groupBy("pickup_zip") \
    .agg(
        avg("trip_distance").alias("avg_trip_distance"),
        avg("trip_minutes").alias("avg_trip_minutes")
    ) \
    .orderBy("pickup_zip")

display(result)

In [0]:
%sql
OPTIMIZE workspace.default.silver_nyc_taxi_trips;